# Curate Materials Project control structures

This notebook builds three deliberately simple control pools from the current matched electride registry:

1. same formula skeleton and space group;
2. isostructural chemical controls based on the same formula skeleton, space group, and similar site count;
3. broad background controls from stable, non-candidate Materials Project structures.

Known matched MP IDs are excluded from every control pool. These are control candidates, not confirmed non-electrides.

In [ ]:
from pathlib import Path
import os
import re
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from mp_api.client import MPRester
from pymatgen.core import Composition

PROJECT_ROOT = Path('/Users/andrewweiland/Andrews_Lab_Research/electride-ml-predictor')
MATCHED_PATH = PROJECT_ROOT / 'data/electrides/final_dataset/inorganic_electride_registry_matched.csv'
OUTPUT_DIR = PROJECT_ROOT / 'data/control/final_dataset'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
load_dotenv(PROJECT_ROOT / '.env')
MP_API_KEY = os.environ.get('MP_API_KEY', '').strip()
if not MP_API_KEY:
    raise ValueError('MP_API_KEY was not found in the project .env file.')

MAX_ISOSTRUCTURAL_PER_SEED = 3
MAX_DOCS_PER_SKELETON = 200
BROAD_CONTROL_COUNT = 300
RANDOM_STATE = 42


In [ ]:
matched_df = pd.read_csv(MATCHED_PATH)
matched_df['mp_id'] = matched_df['mp_id'].astype(str)
candidate_ids = set(matched_df['mp_id'].dropna())
seed_df = (
    matched_df[['mp_id', 'normalized_formula', 'space_group']]
    .drop_duplicates()
    .rename(columns={'mp_id': 'seed_mp_id', 'normalized_formula': 'seed_formula'})
)

def skeleton(formula):
    try:
        return Composition(formula).anonymized_formula
    except Exception:
        return None

seed_df['formula_skeleton'] = seed_df['seed_formula'].map(skeleton)
seed_df = seed_df.dropna(subset=['formula_skeleton', 'space_group'])
print(f'Registry rows: {len(matched_df):,}')
print(f'Unique matched candidate MP IDs: {len(candidate_ids):,}')
print(f'Unique seed formula/space-group rows: {len(seed_df):,}')
seed_df.head()

In [ ]:
FIELDS = ['material_id', 'formula_pretty', 'composition', 'symmetry', 'structure',
          'volume', 'energy_above_hull', 'band_gap', 'is_stable', 'theoretical']

def material_row(doc):
    symmetry = getattr(doc, 'symmetry', None)
    structure = getattr(doc, 'structure', None)
    formula = getattr(doc, 'formula_pretty', None)
    return {
        'mp_id': str(doc.material_id),
        'formula': formula,
        'formula_skeleton': skeleton(formula),
        'space_group': getattr(symmetry, 'symbol', None),
        'space_group_number': getattr(symmetry, 'number', None),
        'crystal_system': getattr(symmetry, 'crystal_system', None),
        'n_sites': len(structure) if structure is not None else None,
        'volume_A3': getattr(doc, 'volume', None),
        'energy_above_hull_eV_per_atom': getattr(doc, 'energy_above_hull', None),
        'band_gap_eV': getattr(doc, 'band_gap', None),
        'is_stable': getattr(doc, 'is_stable', None),
        'theoretical': getattr(doc, 'theoretical', None),
    }

def query_formula_docs(mpr, formula):
    return list(mpr.materials.summary.search(formula=formula, fields=FIELDS))

formula_cache = {}
with MPRester(api_key=MP_API_KEY, mute_progress_bars=True) as mpr:
    for i, formula in enumerate(sorted(seed_df['formula_skeleton'].unique()), 1):
        docs = query_formula_docs(mpr, formula)
        docs = sorted(docs, key=lambda doc: getattr(doc, 'energy_above_hull', None) if getattr(doc, 'energy_above_hull', None) is not None else float('inf'))
        formula_cache[formula] = docs[:MAX_DOCS_PER_SKELETON]
        if i == 1 or i % 25 == 0 or i == seed_df['formula_skeleton'].nunique():
            print(f'Queried {i}/{seed_df["formula_skeleton"].nunique()} formula skeletons')

formula_pool = {}
for formula, docs in formula_cache.items():
    formula_pool[formula] = {str(doc.material_id): doc for doc in docs}
print(f'Formula queries returned {sum(len(v) for v in formula_pool.values()):,} records before deduplication.')

In [ ]:
# Hard controls: same anonymous formula and same reported space group.
same_sg_records = []
isostructural_records = []

for seed in seed_df.itertuples(index=False):
    docs = list(formula_pool.get(seed.formula_skeleton, {}).values())
    same_sg = [doc for doc in docs if getattr(getattr(doc, 'symmetry', None), 'symbol', None) == seed.space_group]
    for doc in same_sg:
        mp_id = str(doc.material_id)
        if mp_id not in candidate_ids:
            row = material_row(doc)
            row.update({'control_type': 'same_formula_skeleton_and_space_group', 'seed_mp_id': seed.seed_mp_id})
            same_sg_records.append(row)

    seed_docs = [doc for doc in same_sg if str(doc.material_id) == seed.seed_mp_id]
    if not seed_docs:
        seed_docs = [doc for doc in docs if str(doc.material_id) == seed.seed_mp_id]
    if not seed_docs:
        continue
    seed_structure = seed_docs[0].structure
    seed_formula = seed_docs[0].formula_pretty
    matches = []
    for doc in docs:
        mp_id = str(doc.material_id)
        if mp_id in candidate_ids or mp_id == seed.seed_mp_id or doc.structure is None:
            continue
        if doc.formula_pretty == seed_formula:
            continue
        if getattr(getattr(doc, 'symmetry', None), 'symbol', None) != seed.space_group:
            continue
        if abs(len(doc.structure) - len(seed_structure)) > max(2, round(0.2 * len(seed_structure))):
            continue
        matches.append(doc)
    for doc in sorted(matches, key=lambda item: str(item.material_id))[:MAX_ISOSTRUCTURAL_PER_SEED]:
        row = material_row(doc)
        row.update({'control_type': 'isostructural_chemical', 'seed_mp_id': seed.seed_mp_id})
        isostructural_records.append(row)

same_sg_df = pd.DataFrame(same_sg_records).drop_duplicates('mp_id')
isostructural_df = pd.DataFrame(isostructural_records).drop_duplicates('mp_id')
print(f'Same formula skeleton + space group controls: {len(same_sg_df):,}')
print(f'Isostructural chemical controls: {len(isostructural_df):,}')

In [ ]:
# Broad background controls: stable MP structures with 2-4 elements and 2-40 sites.
# Only metadata is requested for this large query; structures are fetched after sampling.
METADATA_FIELDS = ['material_id', 'formula_pretty', 'composition', 'symmetry', 'volume',
                   'energy_above_hull', 'band_gap', 'is_stable', 'theoretical']
with MPRester(api_key=MP_API_KEY, mute_progress_bars=True) as mpr:
    broad_docs = list(mpr.materials.summary.search(
        is_stable=True, num_elements=(2, 4), num_sites=(2, 40),
        deprecated=False, fields=METADATA_FIELDS, chunk_size=1000
    ))

known_control_ids = candidate_ids | set(same_sg_df.get('mp_id', [])) | set(isostructural_df.get('mp_id', []))
broad_pool = pd.DataFrame([
    material_row(doc) | {'control_type': 'broad_background', 'seed_mp_id': None}
    for doc in broad_docs if str(doc.material_id) not in known_control_ids
])
if len(broad_pool) > BROAD_CONTROL_COUNT:
    broad_pool = broad_pool.sample(BROAD_CONTROL_COUNT, random_state=RANDOM_STATE)
broad_pool = broad_pool.drop_duplicates('mp_id').reset_index(drop=True)
print(f'Broad background controls: {len(broad_pool):,}')
print(f'Broad query candidates after exclusions: {len(broad_docs):,}')

In [ ]:
# Combine pools and keep provenance when a structure satisfies more than one rule.
control_df = pd.concat([same_sg_df, isostructural_df, broad_pool], ignore_index=True)
control_df['control_type'] = control_df.groupby('mp_id')['control_type'].transform(lambda values: '|'.join(sorted(set(values))))
control_df = control_df.drop_duplicates('mp_id').reset_index(drop=True)
control_df = control_df[~control_df['mp_id'].isin(candidate_ids)].copy()

same_sg_df.to_csv(OUTPUT_DIR / 'same_formula_skeleton_space_group_controls.csv', index=False)
isostructural_df.to_csv(OUTPUT_DIR / 'isostructural_chemical_controls.csv', index=False)
broad_pool.to_csv(OUTPUT_DIR / 'broad_background_controls.csv', index=False)
control_df.to_csv(OUTPUT_DIR / 'control_dataset.csv', index=False)

print('Final control counts:')
print(f'  same skeleton + space group: {len(same_sg_df):,}')
print(f'  isostructural chemical:      {len(isostructural_df):,}')
print(f'  broad background:            {len(broad_pool):,}')
print(f'  unique combined controls:    {len(control_df):,}')
print(f'Wrote control CSVs to {OUTPUT_DIR}')
control_df.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
same_sg_df['space_group'].value_counts().head(20).sort_values().plot.barh(ax=axes[0], color='steelblue')
axes[0].set_title('Same-skeleton controls by space group')
axes[0].set_xlabel('Structures')
control_df['control_type'].str.split('|').explode().value_counts().plot.bar(ax=axes[1], color='darkorange')
axes[1].set_title('Control records by rule')
axes[1].set_ylabel('Structures')
plt.tight_layout()
plt.show()